# Pandas

<svg xmlns="http://www.w3.org/2000/svg" width="480" height="200" viewBox="0 0 600 250">
<rect x="60" y="20" width="480" height="170" rx="14" fill="#ffffff" stroke="#9aa0a6" stroke-width="3"/>
<rect x="60" y="20" width="480" height="38" rx="14" fill="#2d2f33"/>
<rect x="60" y="44" width="480" height="14" fill="#2d2f33"/>
<text x="80" y="47" fill="#f2f2f2" font-family="monospace" font-size="22">patient_id</text>
<text x="270" y="47" fill="#f2f2f2" font-family="monospace" font-size="22">age</text>
<text x="400" y="47" fill="#f2f2f2" font-family="monospace" font-size="22">ward</text>
<line x1="250" y1="20" x2="250" y2="189" stroke="#9aa0a6" stroke-width="2"/>
<line x1="380" y1="20" x2="380" y2="189" stroke="#9aa0a6" stroke-width="2"/>
<line x1="61" y1="102" x2="539" y2="102" stroke="#dadce0" stroke-width="2"/>
<line x1="61" y1="146" x2="539" y2="146" stroke="#dadce0" stroke-width="2"/>
<text x="80" y="90" fill="#1b1b1b" font-family="monospace" font-size="22">P0001</text>
<text x="270" y="90" fill="#1b1b1b" font-family="monospace" font-size="22">66</text>
<text x="400" y="90" fill="#1b1b1b" font-family="monospace" font-size="22">ICU</text>
<text x="80" y="134" fill="#1b1b1b" font-family="monospace" font-size="22">P0002</text>
<text x="270" y="134" fill="#c53030" font-family="monospace" font-size="22">NaN</text>
<text x="400" y="134" fill="#1b1b1b" font-family="monospace" font-size="22">ICU</text>
<text x="80" y="178" fill="#1b1b1b" font-family="monospace" font-size="22">P0003</text>
<text x="270" y="178" fill="#1b1b1b" font-family="monospace" font-size="22">41</text>
<text x="400" y="178" fill="#1b1b1b" font-family="monospace" font-size="22">ONCO</text>
<text x="300" y="235" fill="#2d2f33" font-family="Helvetica, Arial, sans-serif" font-size="26" font-weight="bold" text-anchor="middle">The DataFrame</text>
</svg>

[Chaque question est suivie d'une cellule vide : écrivez-y votre réponse. Utilisez des pandas pour toutes les questions, sauf indication contraire.]

In [1]:
import pandas as pd

## 1. Chargement d'un CSV

Exécutez d'abord la cellule ci-dessous : elle crée l'export du service synthétique `../data/patients.csv` ($240$ patients) si le dossier n'est pas encore là. Tout est synthétique et ensemencé, c'est donc la même exportation pour tout le monde.

In [2]:
from pathlib import Path
from subprocess import run
from sys import executable

if not Path("../data/patients.csv").exists():
    run([executable, "../../scripts/make_data_c01.py", "../data"], check=True)

Lisez le fichier dans un `DataFrame` appelé `df` (`pd.read_csv`), puis regardez ses premières lignes (`.head()`). Un `DataFrame` est un tableau : des colonnes nommées, chacune avec son propre type.

In [3]:
df = pd.read_csv("../data/patients.csv")
df.head()

,patient_id,age,sex,ward,admission,los_days,outcome
0,P0001,66.0,F,ICU,2024-04-21,6.4,transferred
1,P0002,14.0,F,ICU,2024-04-14,10.6,transferred
2,P0003,73.0,M,ICU,2024-07-09,8.6,transferred
3,P0004,6.0,F,PEDIA,2024-09-09,7.0,discharged
4,P0005,63.0,M,ONCO,2024-06-08,9.1,discharged


Imprimez la forme du tableau (`.shape`) et le type de chaque colonne (`.dtypes`).
Quelles colonnes les pandas ont-ils lues sous forme de nombres et lesquelles sous forme de texte (`str`) ?

In [4]:
df.shape, df.dtypes

((240, 7),
 patient_id        str
 age           float64
 sex               str
 ward              str
 admission         str
 los_days      float64
 outcome           str
 dtype: object)

`.info()` donne les deux à la fois, plus le nombre de valeurs qui se trouvent réellement dans chaque colonne. Appelez-le et recherchez la colonne qui manque de quelques valeurs.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   patient_id  240 non-null    str    
 1   age         232 non-null    float64
 2   sex         240 non-null    str    
 3   ward        240 non-null    str    
 4   admission   240 non-null    str    
 5   los_days    240 non-null    float64
 6   outcome     240 non-null    str    
dtypes: float64(2), str(5)
memory usage: 13.3 KB


## 2. Supprimer les valeurs manquantes

Un trou dans le fichier (une cellule vide dans le CSV) est lu comme `NaN` ($8$ âges manquent ici).
Comptez les valeurs manquantes par colonne (`.isna().sum()`).

In [6]:
df.isna().sum()

patient_id    0
age           8
sex           0
ward          0
admission     0
los_days      0
outcome       0
dtype: int64

Construisez `clean`, le tableau sans les lignes qui ont une valeur manquante (`.dropna()`), et comparez le nombre de lignes avant et après (`len`).
`.dropna()` renvoie une nouvelle table, cela ne change pas `df`.

In [7]:
clean = df.dropna()
len(df), len(clean)

(240, 232)

## 3. Changer le type d'une colonne (`.astype`)

`age` a été lu comme un flottant, uniquement à cause des trous. Maintenant qu'ils ont disparu, convertissez-le en un entier : `clean["age"] = clean["age"].astype("int64")`.
Vérifiez avec `.dtypes` que la colonne a changé.

In [8]:
clean["age"] = clean["age"].astype("int64")
clean.dtypes

patient_id        str
age             int64
sex               str
ward              str
admission         str
los_days      float64
outcome           str
dtype: object

Deux autres conversions utiles : `ward` contient $5$ étiquettes répétées, donc `.astype("category")` est à la fois plus rapide et plus petit, et `admission` est une date écrite sous forme de texte, qui `pd.to_datetime` se transforme en une date réelle.
Convertissez les deux, puis imprimez à nouveau `.dtypes`.

In [9]:
clean["ward"] = clean["ward"].astype("category")
clean["admission"] = pd.to_datetime(clean["admission"])
clean.dtypes

patient_id               str
age                    int64
sex                      str
ward                category
admission     datetime64[us]
los_days             float64
outcome                  str
dtype: object

## 4. Sélection d'une colonne

`clean["age"]` est une colonne, un `Series` (un tableau, plus un index).
Imprimez-le, puis sa moyenne, son minimum et son maximum (`.mean()`, `.min()`, `.max()`).

In [10]:
ages = clean["age"]
ages.mean(), ages.min(), ages.max()

(np.float64(50.866379310344826), np.int64(1), np.int64(99))

Sur une colonne de texte ou de catégories, `.value_counts()` compte chaque étiquette.
Combien de séjours par service ? Et par résultat ?

In [11]:
clean["ward"].value_counts(), clean["outcome"].value_counts()

(ward
 ICU       73
 CARDIO    61
 ONCO      38
 PEDIA     34
 GERIA     26
 Name: count, dtype: int64,
 outcome
 discharged     155
 transferred     58
 deceased        19
 Name: count, dtype: int64)

## 5. Sélection d'un sous-ensemble de colonnes

Avec une liste entre crochets, vous obtenez un tableau au lieu d'une colonne : `clean[["patient_id", "ward"]]`.
Construisez `summary`, le tableau avec seulement `patient_id`, `ward`, `age` et `los_days`, et affichez ses premières lignes.

In [12]:
summary = clean[["patient_id", "ward", "age", "los_days"]]
summary.head()

,patient_id,ward,age,los_days
0,P0001,ICU,66,6.4
1,P0002,ICU,14,10.6
2,P0003,ICU,73,8.6
3,P0004,PEDIA,6,7.0
4,P0005,ONCO,63,9.1


Notez la différence : `clean["age"]` est un `Series`, `clean[["age"]]` est un `DataFrame` d'une colonne. Imprimez le `type` des deux.

In [13]:
type(clean["age"]), type(clean[["age"]])

(pandas.Series, pandas.DataFrame)

## 6. Appliquer une fonction à une colonne (`.apply`)

`.apply` exécute une fonction sur chaque valeur d'une colonne et renvoie les résultats sous forme de nouvelle colonne.
Écrivez une fonction `stay_length(days)` qui renvoie `"short"` sous $3$ jours, `"long"` sur $10$ et `"medium"` entre les deux, puis ajoutez la colonne `summary["stay"] = summary["los_days"].apply(stay_length)`.
Comptez le résultat avec `.value_counts()`.

In [14]:
def stay_length(days):
    if days < 3:
        return "short"
    if days > 10:
        return "long"
    return "medium"


summary["stay"] = summary["los_days"].apply(stay_length)
summary["stay"].value_counts()

stay
medium    128
long       54
short      50
Name: count, dtype: int64

`.apply` prend également un lambda, ce qui est pratique pour un one-liner.
Ajoutez une colonne `age_group` contenant `"minor"` ou `"adult"`, avec `summary["age"].apply(lambda a: ...)`.

In [15]:
summary["age_group"] = summary["age"].apply(lambda a: "minor" if a < 18 else "adult")
summary.head()

,patient_id,ward,age,los_days,stay,age_group
0,P0001,ICU,66,6.4,medium,adult
1,P0002,ICU,14,10.6,long,minor
2,P0003,ICU,73,8.6,medium,adult
3,P0004,PEDIA,6,7.0,medium,minor
4,P0005,ONCO,63,9.1,medium,adult


## 7. Sauvegarder au format CSV

Écrivez `summary` à `patients_summary.csv` avec `.to_csv("patients_summary.csv", index=False)`.
`index=False` supprime les numéros de ligne, qui ne sont pas des données : sans cela, le fichier obtient une première colonne supplémentaire sans nom.

In [16]:
summary.to_csv("patients_summary.csv", index=False)

Relisez le fichier (`pd.read_csv`) et vérifiez que vous obtenez le même tableau (`.head()`, `.shape`).
Ouvrez-le également avec le shell, comme lors de la session précédente : `!head -3 patients_summary.csv`.

In [17]:
again = pd.read_csv("patients_summary.csv")
again.shape, again.head()

((232, 6),
   patient_id   ward  age  los_days    stay age_group
 0      P0001    ICU   66       6.4  medium     adult
 1      P0002    ICU   14      10.6    long     minor
 2      P0003    ICU   73       8.6  medium     adult
 3      P0004  PEDIA    6       7.0  medium     minor
 4      P0005   ONCO   63       9.1  medium     adult)